# Interview Problems

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.window import *
from pyspark.sql.types import *
from math import *
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.window import *
from pyspark.sql.types import *
from math import *
import builtins
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

data = [
    ("101", "supplier", "shoes", None, "10:00"),
    ("101", "ml_model", "sneakers", 0.95, "11:00"),
    ("102", "supplier", "shirts", None, "10:00"),
    ("102", "ml_model", "tops", 0.60, "11:00"),
    ("103", "supplier", "trousers", None, "09:00"),
    ("103", "ml_model", "pants", 0.92, "12:00")
]

schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("source", StringType(), False),
    StructField("category", StringType(), True),
    StructField("confidence", DoubleType(), True),
    StructField("updated_at", StringType(), True)
])

df = spark.createDataFrame(data, schema)

display(df)

Find best category based on business rules

In [0]:
sorted_df = (
    df
    .filter((col("confidence").isNull()) | (col("confidence") > 0.90))
    .withColumn("confidence_score",when(coalesce(col("confidence"),lit(.00)) >= 0.90,1).otherwise(0))
    .withColumn("rnk",row_number().over(Window.partitionBy("product_id").orderBy(desc("confidence_score"))))
    .filter(col("rnk") == 1)
    .select("product_id","category")

)

sorted_df.show()

In [0]:
from pyspark.sql import Row

data = [
    Row(product_id=101, tags=["sports", "summer", "sale"]),
    Row(product_id=102, tags=["sports", "running"]),
    Row(product_id=103, tags=["summer", "sale"]),
    Row(product_id=104, tags=None),
    Row(product_id=105, tags=["sports", "sale", "sale"])
]

df = spark.createDataFrame(data)
display(df)

Find the top 2 most common tags across all products.

In [0]:
topsales_df = (
    df.withColumn("tag_count", explode(array_distinct(col("tags"))))
    .groupBy("tag_count")
    .agg(count("tag_count").alias("counttags"))
    .sort(desc("counttags"))
    .limit(2)
)

display(topsales_df)

In [0]:
from pyspark.sql.types import *
from pyspark.sql import Row

schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField(
        "attributes",
        MapType(StringType(), StringType()),
        True
    )
])

data = [
    (101, {"brand": "Nike", "color": "Black", "size": "M"}),
    (102, {"brand": "Adidas", "size": "L"}),
    (103, {"color": "Blue", "size": "S"}),
    (104, {}),
    (105, None)
]

df = spark.createDataFrame(data, schema)
display(df)

A valid product must contain:
brand
color
size

return product,missingattributes


In [0]:
required = [lit("brand"), lit("color"), lit("size")]

exists_df = (
    df
    .withColumn("missing_attributes",
                when((size(col("attributes"))==0) | (col("attributes").isNull()),array(required)).otherwise(array_except(array(required),map_keys(col("attributes"))))
                )
    .filter(size(col("missing_attributes")) > 0)
    .select(col("product_id"), col("missing_attributes"))
)

display(exists_df)


In [0]:
data = [
    ("u1", "2026-09-29 10:00:00", "view"),
    ("u1", "2026-09-29 10:00:00", "click"),
    ("u1", "2026-09-29 10:30:00", "purchase"),
    ("u1", "2026-09-29 11:45:00", "view"),
    ("u1", "2026-09-29 11:50:00", "purchase"),

    ("u2", "2026-09-29 09:00:00", "view"),
    ("u2", "2026-09-29 09:40:00", "view"),
    ("u2", "2026-09-29 09:45:00", "click")
]

df = spark.createDataFrame(
    data,
    ["user_id", "event_time", "event_type"]
)

df = df.withColumn(
    "event_time",
    col("event_time").cast("timestamp")
)

display(df)

Create sessions.
A new session starts when the gap between consecutive events for the same user is greater than 30 minutes.

In [0]:
bestapproach_df = (
    df
    .withColumn("maxsessiontime",max("event_time").over(Window.partitionBy("user_id").orderBy("event_time").rowsBetween(Window.unboundedPreceding,-1)))
    .withColumn("sessiontokens",(col("event_time").cast("long") - col("maxsessiontime").cast("long")) / 60.00)
    .withColumn("sessionidentifier",
                 sum(when(
                    col("sessiontokens") >= 30.00,1
                ).otherwise(0)
                ).over(Window.partitionBy("user_id").orderBy("event_time"))
                )
)

bestapproach_df.display()

In [0]:
session_df = (
    df
    .withColumn("difference", (col("event_time") - lag("event_time", 1).over(Window.partitionBy("user_id").orderBy("event_time"))).cast("long")/60.00)
    .withColumn("sessiontoken",when(col("difference").cast("long") > lit(30.00).cast("long"),1).otherwise(0))
    .withColumn("session",sum("sessiontoken").over(Window.partitionBy("user_id").orderBy("event_time")))
    .groupBy("user_id",concat_ws("_",col("user_id"),col("session")).alias("session"))
    .agg(min("event_time").alias("session_start"),max("event_time").alias("session_end"))
)

session_df.explain("formatted")
session_df.display()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

data = [
    (101, "2026-01-01", "2026-01-30", 100.0),  # A
    (101, "2026-01-05", "2026-01-06", 90.0),   # B
    (101, "2026-01-07", "2026-01-08", 80.0),   # C
    (101, "2026-01-09", "2026-01-10", 70.0),   # D
    (101, "2026-01-20", "2026-01-25", 60.0),   # E overlaps A, 4 rows apart

    (102, "2026-02-01", "2026-02-03", 50.0),
    (102, "2026-02-05", "2026-02-07", 55.0),
    (102, "2026-02-10", "2026-02-12", 60.0)
]

df = spark.createDataFrame(
    data,
    ["product_id", "start_date", "end_date", "price"]
)

df = (
    df
    .withColumn("start_date", F.col("start_date").cast("date"))
    .withColumn("end_date", F.col("end_date").cast("date"))
)

display(df)

Intervals / overlapping price periods

Identify products having overlapping validity periods.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import max

overlapping_df = (
    df
    .withColumn(
        "maxenddate",
        max("end_date").over(
            Window.partitionBy("product_id").orderBy("start_date").rowsBetween(Window.unboundedPreceding, -1)
        )
    )
    .withColumn("isSameValidity",col("start_date")<col("maxenddate"))
)

display(overlapping_df)

## Set operations / anti-joins

Question

Find products that are missing either image data or price data.

In [0]:
products = spark.createDataFrame(
    [
        (101, "shoes"),
        (102, "shirts"),
        (103, "pants"),
        (104, "jackets"),
        (105, "hats")
    ],
    ["product_id", "category"]
)

images = spark.createDataFrame(
    [
        (101,),
        (103,),
        (105,)
    ],
    ["product_id"]
)

prices = spark.createDataFrame(
    [
        (101,),
        (102,),
        (105,)
    ],
    ["product_id"]
)

In [0]:
from pyspark.sql import functions as F

result = (
    products.alias("p")
    .join(
        images.alias("i"),
        on="product_id",
        how="left"
    )
    .join(
        prices.alias("pr"),
        on="product_id",
        how="left"
    )
    .withColumn(
        "missing_image",
        F.col("i.product_id").isNull()
    )
    .withColumn(
        "missing_price",
        F.col("pr.product_id").isNull()
    )
    .filter(
        F.col("missing_image") |
        F.col("missing_price")
    )
    .select(
        F.col("p.product_id"),
        "missing_image",
        "missing_price"
    )
)

display(result)

## Nested JSON + struct + arrays

Question

Build a final dataset with one row per product:

In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F

data = [

    # Product 101
    Row(
        product_id=101,
        supplier=Row(
            supplier_id="SUP-1",
            brand="Nike",
            category_path=["Fashion", "Shoes", "Running"]
        ),
        attributes={
            "color": "Black",
            "material": "Mesh",
            "gender": "Men"
        },
        variants=[
            Row(
                sku="SKU-101-A",
                size="42",
                stock=10,
                price=120.0,
                images=[
                    Row(url="img101_a_main.jpg", type="main", width=1200, height=1500),
                    Row(url="img101_a_side.jpg", type="side", width=800, height=1000)
                ]
            ),
            Row(
                sku="SKU-101-B",
                size="43",
                stock=0,
                price=115.0,
                images=[
                    Row(url="img101_b_main.jpg", type="main", width=700, height=900)
                ]
            )
        ],
        ml_predictions=[
            Row(attribute="color", value="Black", confidence=0.98),
            Row(attribute="category", value="Running Shoes", confidence=0.95),
            Row(attribute="material", value="Synthetic", confidence=0.65)
        ]
    ),

    # Product 102
    Row(
        product_id=102,
        supplier=Row(
            supplier_id="SUP-2",
            brand="Adidas",
            category_path=["Fashion", "Shoes"]
        ),
        attributes={
            "color": "",
            "material": "Leather"
        },
        variants=[
            Row(
                sku="SKU-102-A",
                size="40",
                stock=3,
                price=95.0,
                images=[
                    Row(url="img102_side.jpg", type="side", width=1000, height=1000)
                ]
            ),
            Row(
                sku="SKU-102-B",
                size="41",
                stock=7,
                price=-20.0,
                images=None
            )
        ],
        ml_predictions=[
            Row(attribute="color", value="White", confidence=0.91),
            Row(attribute="category", value="Sneakers", confidence=0.89)
        ]
    ),

    # Product 103
    Row(
        product_id=103,
        supplier=Row(
            supplier_id="SUP-3",
            brand=None,
            category_path=["Fashion", "Clothing", "Trousers"]
        ),
        attributes={
            "color": "Blue",
            "gender": "Women"
        },
        variants=[
            Row(
                sku="SKU-103-A",
                size="M",
                stock=5,
                price=80.0,
                images=[
                    Row(url="img103_main.jpg", type="main", width=500, height=600)
                ]
            )
        ],
        ml_predictions=[
            Row(attribute="brand", value="Levis", confidence=0.93),
            Row(attribute="material", value="Denim", confidence=0.94)
        ]
    ),

    # Product 104
    Row(
        product_id=104,
        supplier=Row(
            supplier_id="SUP-4",
            brand="Puma",
            category_path=[]
        ),
        attributes=None,
        variants=[],
        ml_predictions=[
            Row(attribute="category", value="Sportswear", confidence=0.97)
        ]
    ),

    # Product 105
    Row(
        product_id=105,
        supplier=Row(
            supplier_id="SUP-5",
            brand="Nike",
            category_path=["Fashion", "Shoes", "Lifestyle"]
        ),
        attributes={
            "color": "Red",
            "material": "Synthetic",
            "gender": "Women"
        },
        variants=[
            Row(
                sku="SKU-105-A",
                size="38",
                stock=2,
                price=130.0,
                images=[
                    Row(url="img105_a_main.jpg", type="main", width=1400, height=1600),
                    Row(url="img105_a_main_duplicate.jpg", type="main", width=1000, height=1200)
                ]
            ),
            Row(
                sku="SKU-105-B",
                size="39",
                stock=8,
                price=130.0,
                images=[
                    Row(url="img105_b_side.jpg", type="side", width=1000, height=1300)
                ]
            )
        ],
        ml_predictions=[
            Row(attribute="color", value="Red", confidence=0.99),
            Row(attribute="material", value="Synthetic", confidence=0.96)
        ]
    )
]

df = spark.createDataFrame(data)

display(df)


RETURN:
- product_id
- final_brand
- final_color
- final_material
- final_category
- variant_count
- available_variant_count
- min_valid_price
- main_image_count
- high_quality_main_image_count
- quality_issues

Apply these rules:
- final_brand: use supplier brand if present; otherwise use ML brand prediction if confidence >= 0.90.
- final_color: use ML color if confidence >= 0.90; otherwise use attributes["color"]. Empty string counts as missing.
- final_material: use ML material if confidence >= 0.90; otherwise use supplier attribute material.
- final_category: use ML category if confidence >= 0.90; otherwise use the last element of supplier.category_path.
- variant_count: number of variants.
- available_variant_count: variants where stock > 0.
- min_valid_price: minimum price among variants where price > 0.
- main_image_count: total number of type == "main" images across all variants.
- high_quality_main_image_count: main images where width >= 1000 AND height >= 1200.

In [0]:
class AddtionalFields:
    
    @staticmethod
    def addcolsthreshold(df:DataFrame,arraycol:Column,atttype:StringType,outputfield:Column,threshold:FloatType,supplierattribute:Column) -> DataFrame:
        transformedfield = array_max(transform(filter(arraycol,lambda x:x["attribute"] == atttype),lambda x:struct(x["confidence"],x["value"])))
        supptype = df.select(col(supplierattribute)).schema[0].dataType

        if isinstance(supptype,ArrayType):
            supplier = element_at(col(supplierattribute),-1)
        else:
            supplier = col(supplierattribute)

        return df.withColumn(
            outputfield,
            when(transformedfield["confidence"] >= threshold,transformedfield["value"])
            .when((supplier.isNotNull()) & (supplier != ''),supplier)
        )

cleaned_df = (
    df
    .transform(
        AddtionalFields.addcolsthreshold,
        arraycol="ml_predictions",
        atttype="brand",
        outputfield="final_brand",
        threshold=0.90,
        supplierattribute="supplier.brand",
    )
    .transform(
        AddtionalFields.addcolsthreshold,
        arraycol="ml_predictions",
        atttype="color",
        outputfield="final_color",
        threshold=0.90,
        supplierattribute="attributes.color",
    )
    .transform(
        AddtionalFields.addcolsthreshold,
        arraycol="ml_predictions",
        atttype="material",
        outputfield="final_material",
        threshold=0.90,
        supplierattribute="attributes.material",
    )
    .transform(
        AddtionalFields.addcolsthreshold,
        arraycol="ml_predictions",
        atttype="category",
        outputfield="final_category",
        threshold=0.90,
        supplierattribute="supplier.category_path",
    )
    .withColumn("variantcount",size(array_distinct(col("variants")["sku"])))
    .withColumn("available_variant_count",size(array_distinct(filter(col("variants"),lambda x:x["stock"]>0))))
    .withColumn("min_valid_price",coalesce(array_min(filter(col("variants"),lambda x:x["price"]>0)["price"]),lit(0)))
    .withColumn("main_image_count",size(filter(flatten(transform(col("variants"),lambda x:coalesce(x["images"],array()))),lambda x:x["type"]=="main")))
    .withColumn("high_quality_main_image_count",size(filter(col("variants"),lambda x:exists(x["images"],lambda y: (y["type"]=="main") & (y["width"]>1000) & (y["height"]>=1200)))))
    .select("product_id","final_brand","final_color","final_material","final_category","variantcount","available_variant_count","min_valid_price","main_image_count","high_quality_main_image_count")
)


cleaned_df.display()

## Adding rule category

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import date

product_schema = T.StructType([
    T.StructField("product_id", T.LongType(), False),
    T.StructField("market", T.StringType(), False),

    T.StructField(
        "category_map",
        T.MapType(T.StringType(), T.StringType()),
        True
    ),

    T.StructField(
        "attributes",
        T.MapType(T.StringType(), T.StringType()),
        True
    )
])

product_data = [
    (
        101,
        "DE",
        {
            "department": "Fashion",
            "category": "Shoes",
            "subcategory": "Running"
        },
        {
            "material": "leather",
            "color": "black",
            "sole_type": "rubber",
            "title": "Performance Running Shoe"
        }
    ),

    (
        102,
        "DE",
        {
            "department": "Fashion",
            "category": "Shoes",
            "subcategory": "Running"
        },
        {
            "color": "white",
            "title": "Runner"
        }
    ),

    (
        103,
        "IE",
        {
            "department": "Fashion",
            "category": "Jeans",
            "subcategory": "Slim"
        },
        {
            "waist_size": "32",
            "color": "blue"
        }
    ),

    (
        104,
        "DE",
        {
            "department": "Fashion",
            "category": "Jeans",
            "subcategory": "Slim"
        },
        {
            "color": "black"
        }
    ),

    (
        105,
        "DE",
        {
            "department": "Fashion",
            "category": "Shoes",
            "subcategory": "Casual"
        },
        {
            "material": "",
            "color": "red",
            "title": "Casual Leather Shoe"
        }
    )
]

products_df = spark.createDataFrame(
    product_data,
    schema=product_schema
)

display(products_df)

In [0]:
rule_schema = T.StructType([
    T.StructField("rule_id", T.StringType(), False),
    T.StructField("rule_version", T.IntegerType(), False),

    T.StructField("category_level", T.StringType(), False),
    T.StructField("category_value", T.StringType(), False),

    T.StructField("market", T.StringType(), False),

    T.StructField("attribute_name", T.StringType(), False),
    T.StructField("rule_type", T.StringType(), False),
    T.StructField("expected_value", T.StringType(), True),

    T.StructField("severity", T.StringType(), False),

    T.StructField("effective_from", T.DateType(), False),
    T.StructField("effective_to", T.DateType(), True),

    T.StructField("enabled", T.BooleanType(), False)
])

rule_data = [
    # Shoes must have material
    (
        "R001", 1,
        "category", "Shoes",
        "*",
        "material", "REQUIRED", None,
        "ERROR",
        date(2026, 1, 1), None,
        True
    ),

    # Jeans must have waist_size
    (
        "R002", 1,
        "category", "Jeans",
        "*",
        "waist_size", "REQUIRED", None,
        "ERROR",
        date(2026, 1, 1), None,
        True
    ),

    # Running products must have sole_type
    (
        "R003", 1,
        "subcategory", "Running",
        "*",
        "sole_type", "REQUIRED", None,
        "WARNING",
        date(2026, 1, 1), None,
        True
    ),

    # German shoes can only have these colors
    (
        "R004", 1,
        "category", "Shoes",
        "DE",
        "color", "IN_SET", "black,white,blue",
        "WARNING",
        date(2026, 1, 1), None,
        True
    ),

    # Shoe title must be at least 10 chars
    (
        "R005", 1,
        "category", "Shoes",
        "*",
        "title", "MIN_LENGTH", "10",
        "WARNING",
        date(2026, 1, 1), None,
        True
    )
]

rules_df = spark.createDataFrame(
    rule_data,
    schema=rule_schema
)

display(rules_df)

In [0]:
active_rules = (
    rules_df
    .filter(F.col("enabled"))
    .filter(
        (F.col("effective_from") <= F.current_date())
        &
        (
            F.col("effective_to").isNull()
            | (F.current_date() < F.col("effective_to"))
        )
    )
)

In [0]:
p = products_df.alias("p")
r = F.broadcast(active_rules).alias("r")

matched_df = (
    p.join(
        r,
        (
            F.element_at(
                F.col("p.category_map"),
                F.col("r.category_level")
            )
            == F.col("r.category_value")
        )
        &
        (
            (F.col("r.market") == "*")
            | (F.col("p.market") == F.col("r.market"))
        ),
        "inner"
    )
    .select(
        F.col("p.product_id").alias("product_id"),
        F.col("p.market").alias("product_market"),
        F.col("p.attributes").alias("attributes"),
        F.col("p.category_map").alias("catefory"),
        F.col("r.rule_id").alias("rule_id"),
        F.col("r.rule_version").alias("rule_version"),
        F.col("r.category_level").alias("category_level"),
        F.col("r.category_value").alias("category_value"),
        F.col("r.attribute_name").alias("attribute_name"),
        F.col("r.rule_type").alias("rule_type"),
        F.col("r.expected_value").alias("expected_value"),
        F.col("r.severity").alias("severity")
    )
)

matched_df = (
    matched_df
    .withColumn(
        "actual_value",
        F.element_at(
            F.col("attributes"),
            F.col("attribute_name")
        )
    )
)

matched_df.display()

In [0]:
evaluated_df = (
    matched_df
    .withColumn(
        "is_violation",

        F.when(
            F.col("rule_type") == "REQUIRED",
            F.col("actual_value").isNull()
            | (F.trim(F.col("actual_value")) == "")
        )

        .when(
            F.col("rule_type") == "EQUALS",
            ~F.col("actual_value").eqNullSafe(
                F.col("expected_value")
            )
        )

        .when(
            F.col("rule_type") == "MIN_LENGTH",
            F.col("actual_value").isNull()
            |
            (
                F.length(F.trim(F.col("actual_value")))
                <
                F.col("expected_value").cast("int")
            )
        )

        .when(
            F.col("rule_type") == "IN_SET",
            F.col("actual_value").isNull()
            |
            ~F.array_contains(
                F.transform(
                    F.split(F.col("expected_value"), ","),
                    lambda x: F.trim(x)
                ),
                F.col("actual_value")
            )
        )

        .otherwise(False)
    )
)

evaluated_df.display()

In [0]:
violations_df = (
    evaluated_df
    .filter(F.col("is_violation"))
    .select(
        "product_id",
        "rule_id",
        "rule_version",
        "category_level",
        "category_value",
        "attribute_name",
        "rule_type",
        "expected_value",
        "actual_value",
        "severity"
    )
    .withColumn(
        "evaluated_at",
        F.current_timestamp()
    )
)

display(violations_df)

### for production

In [0]:
rulesets_df = (
    rules_df
    .filter(F.col("enabled"))
    .groupBy(
        "category_value",
        "market"
    )
    .agg(
        F.collect_list(
            F.struct(
                "rule_id",
                "rule_version",
                "attribute_name",
                "rule_type",
                "expected_value",
                "severity"
            )
        ).alias("rules")
    )
)

rulesets_df.display()

products_with_rules = (
    products_df.alias("p")
    .join(
        rulesets_df.alias("r"),
        (
            F.element_at(
                F.col("p.category_map"),
                F.lit("category")
            )
            == F.col("r.category_value")
        )
        &
        (
            (F.col("p.market") == F.col("r.market"))
        ),
        "left"
    )
)

products_with_rules.display()

In [0]:
result = (
    products_with_rules
    .withColumn(
        "violations",
            F.transform(
                F.col("rules"),

                lambda r: F.struct(
                    r["rule_id"].alias("rule_id"),
                    r["severity"].alias("severity"),
                    r["attribute_name"].alias("attribute_name"),

                    F.element_at(
                        F.col("attributes"),
                        r["attribute_name"]
                    ).alias("actual_value"),

                    (
                        F.when(r["rule_type"] == "IN_SET", ~F.array_contains(F.split(r["expected_value"],","), F.element_at(
                        F.col("attributes"),
                        r["attribute_name"]
                    )))

                    ).alias("is_violation")
                )
            )
    )
)

result.display()